# 臺大288校補查品質

參考文字、明列當期的接待校事實、臺大正式甄選與實際交換校區分開。4032列有狀態不代表4032列均核實。執行前先重建候選資料與audit輸出。

In [1]:
from pathlib import Path
import json
from collections import Counter
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sources/ntu/completion-2026-10-11.json").exists())
snapshot = json.loads((root / "sources/ntu/completion-2026-10-11.json").read_text())
review = json.loads((root / "docs/ntu-completion-20261011/all-school-review.json").read_text())
assert len(snapshot["schools"]) == 288
assert len(review) == 288 * 14
assert len({(r["portal_id"], r["field"]) for r in review}) == len(review)
print("School/field grain:", len(review))

School/field grain: 4032


In [2]:
before = json.loads((root / "docs/ntu-completion-20261011/before/summary.json").read_text())
after = json.loads((root / "docs/ntu-completion-20261011/review-summary.json").read_text())
for field in before["fields"]:
    print(field, before["fields"][field]["public_evidence_gap"], "->", after["fields"][field]["public_evidence_gap"])
assert after["fully_verified_target_cycle_schools"] == 0
print(Counter(r["issue_code"] for r in review))

home_selection 0 -> 0
eligibility 168 -> 165
language 180 -> 179
quota 282 -> 282
deadlines 194 -> 191
documents 186 -> 182
calendar 174 -> 172
courses 157 -> 151
housing 162 -> 150
cost 171 -> 168
insurance_visa 167 -> 161
life_sources 164 -> 147
coordinates 19 -> 18
photos 22 -> 9
Counter({'technical_transport_failure_prevents_check': 1383, 'prior_reference_target_year_or_subfields_unconfirmed': 1096, 'asset_or_home_fact_checked_exchange_assignment_still_separate': 843, 'official_list_blank_as_of_check': 282, 'discovery_lead_requires_semantic_review': 131, 'reviewed_fact_in_original_scope': 119, 'access_limited_no_verified_field_text': 106, 'not_found_in_bounded_search_not_proof_unannounced': 40, 'campus_identity_or_licensed_asset_unresolved': 27, 'version_conflict_requires_confirmation': 4, 'officially_stated_future_update_pending': 1})


In [3]:
observations = [r for school in snapshot["schools"] for r in school["observations"]]
assert all(r["selection_rule_confirmed"] is False for r in observations)
assert all(r["quote_source_id"] in snapshot["public_source_reviews"] for r in observations)
print("Manually reviewed observations:", len(observations))
print("Original applicability:", Counter(r["applicability_status"] for r in observations))
print("Conflict observations:", [(school["school_name"], r["field"]) for school in snapshot["schools"] for r in school["observations"] if r["applicability_status"] == "version_conflict"])

Manually reviewed observations: 130
Original applicability: Counter({'undated_public_reference': 86, 'target_year_explicit': 33, 'older_year_reference': 7, 'version_conflict': 3, 'explicitly_pending_announcement': 1})
Conflict observations: [('魁北克大學蒙特婁分校', 'documents'), ('魁北克大學蒙特婁分校', 'deadlines'), ('優密歐大學', 'calendar')]


來源全文保留研究工作區；公開候選JSON只保留短引句、人工摘要、SHA與定位。官方網頁快取讀取另標方法，不冒稱保存最新HTTP正文。技術503、robots限制、提名後資訊與確實待公告各自保留。